In [1]:
# load your insider trading data
import pandas as pd
df_events = pd.read_csv('sec_filings.csv', sep=';')
df_filtered = df_events[['Filing Date', 'Ticker', 'Company Name','Trade Type']].copy()
df_filtered['Filing Date'] = df_filtered['Filing Date'].apply(lambda x: x.split(' ')[0])

# drop duplicates based on filing date and ticker
df_unique = df_filtered.drop_duplicates(subset=['Filing Date', 'Ticker', 'Trade Type'])

# display the new dataframe
print("\nNew DataFrame with unique Filing Date and Ticker combinations:")
print(df_unique)
print(f"New shape: {df_unique.shape}")

df_unique.to_csv('unique_event_filings.csv', index=False)


New DataFrame with unique Filing Date and Ticker combinations:
    Filing Date Ticker                Company Name   Trade Type
0    19.11.2024   AAPL                  Apple Inc.     S - Sale
2    18.12.2024   AAPL                  Apple Inc.     S - Sale
3    04.02.2025   AAPL                  Apple Inc.  S - Sale+OE
4    03.04.2025   AAPL                  Apple Inc.  S - Sale+OE
7    25.04.2025   AAPL                  Apple Inc.     S - Sale
..          ...    ...                         ...          ...
712  24.04.2025     VZ  Verizon Communications Inc     S - Sale
713  25.04.2025     VZ  Verizon Communications Inc  S - Sale+OE
714  01.05.2025     VZ  Verizon Communications Inc     S - Sale
715  08.05.2025     VZ  Verizon Communications Inc     S - Sale
716  09.05.2025     VZ  Verizon Communications Inc  S - Sale+OE

[492 rows x 4 columns]
New shape: (492, 4)


# Notebook 02 — Traditional Abnormal Return Calculation

This is where the abnormal returns (ARs) are computed using the classic market model approach from the event study literature (see MacKinlay, 1997). The idea is pretty straightforward:

1. For each event, a 200-day estimation window (days -220 to -21 relative to the event) is used to estimate the market model parameters alpha and beta via OLS regression.
2. Then, in the event window (days -20 to +20), the abnormal return is computed as: `AR_t = R_it - (alpha + beta * R_mt)`

So the AR is basically the stock's actual return minus what would be expected based on how the market moved that day.

**Inputs:**
- `sec_filings.csv` / `unique_event_filings.csv`: the event dates
- `portfolio_returns.csv`: daily stock returns
- `market_returns.csv`: daily NASDAQ returns

**Outputs:**
- `event_study_metadata.csv`: one row per event with alpha, beta, event type
- `event_study_abnormal_returns.csv`: daily AR series for every event in the [-20, +20] window

In [2]:
import pandas as pd
import numpy as np
from datetime import timedelta
import statsmodels.api as sm

# load your insider trading data
df_events = pd.read_csv('unique_event_filings.csv', sep=',')
print(df_events.columns)
df_events['Filing Date'] = pd.to_datetime(df_events['Filing Date'], format='%d.%m.%Y')

# load portfolio returns (wide format) and market returns
df_portfolio = pd.read_csv('portfolio_returns.csv', parse_dates=['date'])
df_market = pd.read_csv('market_returns.csv', parse_dates=['date'])


def _build_trading_calendar(market_df):
    return np.sort(market_df["date"].dropna().unique())

def _trading_day_offset(trading_cal, event_date, offset):
    idx = np.searchsorted(trading_cal, np.datetime64(event_date), side="right") - 1
    if idx < 0:
        idx = 0
    target = idx + offset
    target = max(0, min(target, len(trading_cal) - 1))
    return trading_cal[target]

trading_cal = _build_trading_calendar(df_market)


def estimate_market_model(ticker, portfolio_df, market_df, event_date, window=(-220, -21)):
    # date range for estimation window using trading day offsets
    est_start = _trading_day_offset(trading_cal, event_date, window[0])
    est_end = _trading_day_offset(trading_cal, event_date, window[1])
    
    # filter data for estimation window
    est_data = portfolio_df[(portfolio_df['date'] >= est_start) & 
                            (portfolio_df['date'] <= est_end)].copy()
    
    # skip if ticker not in portfolio data
    if ticker not in est_data.columns:
        return np.nan, np.nan
    
    # merge with market data
    market_data = market_df[(market_df['date'] >= est_start) & 
                           (market_df['date'] <= est_end)].copy()
    merged = pd.merge(est_data[['date', ticker]], market_data[['date', 'market_return']], on='date')
    
    # skip if not enough data points
    if len(merged) < 200:
        print(f"Warning: Not enough data points for {ticker} in estimation window, skipping.")
        return np.nan, np.nan
    
    # estimate market model
    X = sm.add_constant(merged['market_return'])
    y = merged[ticker]
    model = sm.OLS(y, X).fit()
    return model.params['const'], model.params['market_return']

def calculate_ARs(ticker, portfolio_df, market_df, event_date, alpha, beta, window=(-20, 20)):
    # date range for event window using trading-day offsets
    event_start = _trading_day_offset(trading_cal, event_date, window[0])
    event_end = _trading_day_offset(trading_cal, event_date, window[1])
    
    # filter data for event window
    event_data = portfolio_df[(portfolio_df['date'] >= event_start) & 
                             (portfolio_df['date'] <= event_end)].copy()
    
    if ticker not in event_data.columns:
        return pd.DataFrame()
    
    # merge with market data
    market_data = market_df[(market_df['date'] >= event_start) & 
                           (market_df['date'] <= event_end)].copy()
    merged = pd.merge(event_data[['date', ticker]], market_data[['date', 'market_return']], on='date')
    
    # calculate abnormal returns
    merged['AR'] = merged[ticker] - (alpha + beta * merged['market_return'])
    ARs = merged[['date', 'AR']].copy()
    
    return ARs

# process each event
metadata_rows = []
all_ARs = pd.DataFrame(columns=['ticker', 'event_date', 'date', 'AR'])

for idx, row in df_events.iterrows():
    ticker = row['Ticker']
    event_date = row['Filing Date']
    event_type = row['Trade Type']
    
    # skip if ticker not in our portfolio data
    if ticker not in df_portfolio.columns:
        print(f"Warning: {ticker} not found in portfolio data, skipping.")
        continue
    
    alpha, beta = estimate_market_model(ticker, df_portfolio, df_market, event_date)
    
    if np.isnan(alpha) or np.isnan(beta):
        print(f"Warning: Could not estimate market model for {ticker} at {event_date}, skipping.")
        continue
    
    ARs = calculate_ARs(ticker, df_portfolio, df_market, event_date, alpha, beta)
    
    if ARs.empty:
        print(f"Warning: No event window data for {ticker} at {event_date}, skipping.")
        continue

    metadata_rows.append({
        'ticker': ticker,
        'event_date': event_date,
        'event_type': event_type,
        'alpha': alpha,
        'beta': beta,
    })
    
    # record the AR series along with ticker and event date
    ar_data = ARs.copy()
    ar_data['ticker'] = ticker
    ar_data['event_date'] = event_date
    all_ARs = pd.concat([all_ARs, ar_data[['ticker', 'event_date', 'date', 'AR']]], ignore_index=True)



metadata_df = pd.DataFrame(metadata_rows)
if not metadata_df.empty:
    print(f"\nProcessed {len(metadata_df)} events successfully.")
    print("\nExample of last event's results:")
    last_row = metadata_df.iloc[-1]
    print(f"Ticker: {last_row['ticker']}")
    print(f"Event Date: {last_row['event_date']}")
    print(f"Market Model: Alpha = {last_row['alpha']:.6f}, Beta = {last_row['beta']:.6f}")
    print("\nAbnormal Returns (first 5 days):")
    print(ARs.head())
    metadata_df.to_csv('event_study_metadata.csv', index=False)
    all_ARs.to_csv('event_study_abnormal_returns.csv', index=False)
else:
    print("No events were successfully processed.")

Index(['Filing Date', 'Ticker', 'Company Name', 'Trade Type'], dtype='str')

Processed 492 events successfully.

Example of last event's results:
Ticker: VZ
Event Date: 2025-05-09 00:00:00
Market Model: Alpha = 0.000671, Beta = -0.067533

Abnormal Returns (first 5 days):
        date        AR
0 2025-04-10  0.013192
1 2025-04-11  0.019590
2 2025-04-14  0.012795
3 2025-04-15  0.003359
4 2025-04-16 -0.022302


All 492 events were processed successfully, so every stock had sufficient data in its estimation window. The resulting AR series spans 41 trading days per event (t=-20 to t=+20), totalling 19,474 daily AR observations. These AR results will be used in the next stept to calculate the CAR. Analysis of the AR results can be found in the `04_cross_sectional_analysis.ipynb`.